# 07 · Implementación Gold — Modelo Estrella

**Objetivo:** construir la capa **Gold** (dimensiones y hechos del modelo estrella propuesto en el Notebook 06) a partir de la capa Silver.

**Contenido de este notebook (por secciones):**
1. **Generación de esquemas Silver V2** — documentación de la estructura real de los datos limpios.
2. **Construcción de dimensiones Gold** (con claves surrogate `SK_*`).
3. **Construcción de hechos Gold** (con FKs).
4. **Validación del modelo Gold**.

**Entradas:**
- `data/Silver/ingresantes_clean_v2.parquet`
- `data/Silver/matriculados_clean_v2.parquet`

**Salidas:**
- `data/schemas/ingresantes_schema_v2.json`, `data/schemas/matriculados_schema_v2.json`
- `data/Gold/dim_*.parquet` (5 dimensiones)
- `data/Gold/fact_*.parquet` (2 hechos)

In [ ]:
# Configuración: límite de hilos ANTES de importar Polars (12 núcleos)
import os
os.environ['POLARS_MAX_THREADS'] = '12'

import gc
import json
from datetime import datetime
from pathlib import Path

import polars as pl

pl.Config.set_streaming_chunk_size(32 * 1024 * 1024)  # 32 MB por lote de streaming

print('polars', pl.__version__)
print('hilos activos:', pl.thread_pool_size())


def rss_actual_gb():
    '''RSS actual del proceso en GB (Linux, /proc/self/statm).'''
    try:
        with open('/proc/self/statm', encoding='utf-8') as fh:
            paginas = int(fh.read().split()[1])
        return paginas * os.sysconf('SC_PAGE_SIZE') / (1024**3)
    except (OSError, ValueError, IndexError):
        return float('nan')


print(f'RSS inicial: {rss_actual_gb():.2f} GB')
print()

# Rutas del proyecto (misma detección automática que los notebooks 01-06)
current_dir = Path.cwd()
if (current_dir / 'data').exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / 'data').exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError('No se encontró la carpeta data. Ejecuta desde la raíz o desde notebooks/.')

SILVER = PROJECT_ROOT / 'data' / 'Silver'
GOLD = PROJECT_ROOT / 'data' / 'Gold'
SCHEMAS = PROJECT_ROOT / 'data' / 'schemas'
ING_V2 = SILVER / 'ingresantes_clean_v2.parquet'
MAT_V2 = SILVER / 'matriculados_clean_v2.parquet'

assert ING_V2.exists(), f'No existe {ING_V2.name}. Ejecuta primero 04_limpieza_ingresantes.ipynb.'
assert MAT_V2.exists(), f'No existe {MAT_V2.name}. Ejecuta primero 04_limpieza_matriculados.ipynb.'
assert SCHEMAS.exists(), f'No existe la carpeta {SCHEMAS}.'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('Silver:', SILVER)
print('Gold:', GOLD)
print('Schemas:', SCHEMAS)

---
## SECCIÓN 1 · Generación de esquemas Silver V2

Documenta la estructura real de los datasets **limpios** (Silver V2), a diferencia de los esquemas originales (`*_schema.json`) que reflejan el **Bronce**.

Para cada dataset se guarda: `dataset`, `version`, `fecha_generacion`, `clave_natural`, `columnas`, `dtypes`, `n_filas`, `transformaciones` y métricas de calidad. **No se sobrescriben** los esquemas de Bronce.

In [ ]:
# Claves naturales definitivas (validadas en los notebooks 04 y 05)
CLAVE_ING = ['CODIGO_INEI', 'GUID_PERSONA', 'CODIGO_SIU_PROGRAMA']
K5 = ['PERIODO_ESTANDARIZADO', 'CODIGO_INEI', 'CODIGO_SIU_PROGRAMA', 'CODIGO_LOCAL', 'GUID_PERSONA']

# Carga de los datasets limpios (V2): Ingresantes completo (pequeño), Matriculados en modo lazy
ing = pl.read_parquet(ING_V2)
mat = pl.scan_parquet(MAT_V2)

mat_schema = mat.collect_schema()
mat_n = mat.select(pl.len()).collect().item()

print('ingresantes_clean_v2:', ing.shape)
print(f'matriculados_clean_v2: {mat_n:,} filas · {len(mat_schema.names())} columnas (lazy)')
print(f'RSS tras carga: {rss_actual_gb():.2f} GB')

In [ ]:
# --- Ingresantes: ingresantes_schema_v2.json ---
schema_ing = {
    'dataset': 'ingresantes',
    'version': 'v2',
    'fecha_generacion': datetime.now().isoformat(),
    'clave_natural': CLAVE_ING,
    'columnas': ing.columns,
    'dtypes': {c: str(t) for c, t in ing.schema.items()},
    'n_filas': len(ing),
    'transformaciones': [
        'Eliminación de duplicados exactos (293)',
        'Imputación de grupos de carrera (mapa + -1)',
        'Imputación de DEPARTAMENTO_NACIMIENTO y NACIONALIDAD → NO ESPECIFICADO',
        'Imputación de ANIO_NACIMIENTO con mediana',
        'Normalización de textos (MAYÚSCULAS, sin tildes, sin espacios dobles)',
        'Compactación de discapacidad en TIENE_DISCAPACIDAD',
    ],
    'duplicados_eliminados': 293,
}

path_ing_schema = SCHEMAS / 'ingresantes_schema_v2.json'
with open(path_ing_schema, 'w', encoding='utf-8') as fh:
    json.dump(schema_ing, fh, indent=2, ensure_ascii=False)

print(f'OK: creado {path_ing_schema}')
print(f'    versión {schema_ing["version"]} · {schema_ing["n_filas"]:,} filas · {len(schema_ing["columnas"])} columnas')

In [ ]:
# --- Matriculados: matriculados_schema_v2.json ---
schema_mat = {
    'dataset': 'matriculados',
    'version': 'v2',
    'fecha_generacion': datetime.now().isoformat(),
    'clave_natural': K5,
    'columnas': mat_schema.names(),
    'dtypes': {c: str(t) for c, t in mat_schema.items()},
    'n_filas': mat_n,
    'transformaciones': [
        'Imputación de grupos de carrera (mapa + -1)',
        'Imputación de DEPARTAMENTO_NACIMIENTO y NACIONALIDAD → NO ESPECIFICADO',
        'Imputación de ANIO_PERIODO_INGRESO desde PERIODO_ESTANDARIZADO',
        'Normalización de textos (MAYÚSCULAS, sin tildes, sin espacios dobles)',
        'Compactación de discapacidad en TIENE_DISCAPACIDAD',
    ],
    'duplicados_exactos_identificados': 6639,
}

path_mat_schema = SCHEMAS / 'matriculados_schema_v2.json'
with open(path_mat_schema, 'w', encoding='utf-8') as fh:
    json.dump(schema_mat, fh, indent=2, ensure_ascii=False)

print(f'OK: creado {path_mat_schema}')
print(f'    versión {schema_mat["version"]} · {schema_mat["n_filas"]:,} filas · {len(schema_mat["columnas"])} columnas')

In [ ]:
# --- Validación de los archivos de esquema generados ---
print('VALIDACIÓN DE ESQUEMAS V2')
print('=' * 72)
for nombre, path, dataset in [
    ('Ingresantes', path_ing_schema, 'ingresantes'),
    ('Matriculados', path_mat_schema, 'matriculados'),
]:
    with open(path, encoding='utf-8') as fh:
        data = json.load(fh)
    ok = (
        data['dataset'] == dataset
        and data['version'] == 'v2'
        and isinstance(data['columnas'], list)
        and isinstance(data['dtypes'], dict)
        and isinstance(data['clave_natural'], list)
        and data['n_filas'] > 0
    )
    print(f'{nombre}:')
    print(f'  archivo:   {path.name}')
    print(f'  dataset:   {data["dataset"]} · versión {data["version"]}')
    print(f'  filas:     {data["n_filas"]:,}')
    print(f'  columnas:  {len(data["columnas"])}')
    print(f'  clave:     {" + ".join(data["clave_natural"])}')
    print(f'  dtypes:    {len(data["dtypes"])} tipos · transformaciones: {len(data["transformaciones"])}')
    print(f'  OK estructura: {ok}')
    print()
print('=' * 72)
print(f'JSONs generados: {path_ing_schema.name}, {path_mat_schema.name}')

---
## SECCIÓN 2 · Construcción de dimensiones Gold (con SK)

Se generan las **5 dimensiones** del modelo estrella con claves surrogate (`SK_*`), a partir de catálogos construidos con `scan_parquet` + `unique()` (sin OOM). Se garantiza unicidad sobre la clave natural (`.unique(subset=...)`) antes de asignar el `SK_*` con `with_row_index(offset=1)`.

Se guardan en `data/Gold/`:
- `dim_universidad.parquet` — unifica `LICENCIADO`/`LICENCIA` → `ESTADO_LICENCIAMIENTO`
- `dim_programa.parquet`
- `dim_periodo.parquet` — semestres (MAT) + anuales con `SEMESTRE=NULL` (ING)
- `dim_ubicacion.parquet` — departamento+provincia, `Region_Sur` derivada
- `dim_local.parquet` — solo Matriculados

In [ ]:
GOLD.mkdir(exist_ok=True)
DEPARTAMENTOS_SUR = ['AREQUIPA', 'CUSCO', 'TACNA', 'PUNO', 'MOQUEGUA', 'APURIMAC']

# --- DimUniversidad: unión ING (LICENCIADO) + MAT (LICENCIA) → ESTADO_LICENCIAMIENTO ---
dim_univ = (
    pl.concat([
        pl.scan_parquet(ING_V2)
        .select(['CODIGO_INEI', 'NOMBRE_ENTIDAD', 'TIPO_ENTIDAD', 'TIPO_GESTION', 'LICENCIADO', 'TIPO_CONSTITUCION'])
        .rename({'LICENCIADO': 'LICENCIA'})
        .unique()
        .collect(),
        pl.scan_parquet(MAT_V2)
        .select(['CODIGO_INEI', 'NOMBRE_ENTIDAD', 'TIPO_ENTIDAD', 'TIPO_GESTION', 'LICENCIA', 'TIPO_CONSTITUCION'])
        .unique()
        .collect(),
    ])
    .unique()
    .unique(subset=['CODIGO_INEI'])
    .sort('CODIGO_INEI')
    .with_columns(
        pl.when(pl.col('LICENCIA') == 'LICENCIADA').then(pl.lit('LICENCIADO'))
        .when(pl.col('LICENCIA') == 'LEY DE CREACION').then(pl.lit('LEY DE CREACION'))
        .otherwise(pl.lit('NO LICENCIADO'))
        .alias('ESTADO_LICENCIAMIENTO')
    )
    .drop('LICENCIA')
    .with_row_index('SK_Universidad', offset=1)
)
dim_univ.write_parquet(GOLD / 'dim_universidad.parquet')
print(f'DimUniversidad: {dim_univ.height:,} filas → dim_universidad.parquet')

# --- DimPrograma: unión ING + MAT ---
cols_prog = ['CODIGO_SIU_PROGRAMA', 'NOMBRE_PROGRAMA', 'CODIGO_GRUPO_1', 'NOMBRE_GRUPO_1', 'CODIGO_GRUPO_3', 'NOMBRE_GRUPO_3', 'NIVEL_ACADEMICO']
dim_prog = (
    pl.concat([
        pl.scan_parquet(ING_V2).select(cols_prog).unique().collect(),
        pl.scan_parquet(MAT_V2).select(cols_prog).unique().collect(),
    ])
    .unique()
    .unique(subset=['CODIGO_SIU_PROGRAMA'])
    .sort('CODIGO_SIU_PROGRAMA')
    .with_row_index('SK_Programa', offset=1)
)
dim_prog.write_parquet(GOLD / 'dim_programa.parquet')
print(f'DimPrograma: {dim_prog.height:,} filas → dim_programa.parquet')

# --- DimPeriodo: semestres (MAT) + anuales (ING, SEMESTRE=NULL) ---
dim_periodo_sem = (
    pl.scan_parquet(MAT_V2).select('PERIODO_ESTANDARIZADO').unique().collect()
    .with_columns([
        pl.col('PERIODO_ESTANDARIZADO').str.slice(0, 4).cast(pl.Int64).alias('ANIO'),
        pl.col('PERIODO_ESTANDARIZADO').str.slice(5, 1).cast(pl.Int64).alias('SEMESTRE'),
        pl.lit('SEMESTRAL').alias('TIPO_PERIODO'),
    ])
    .select(['ANIO', 'SEMESTRE', pl.col('PERIODO_ESTANDARIZADO').alias('LABEL_PERIODO'), 'TIPO_PERIODO'])
)
dim_periodo_an = (
    pl.scan_parquet(ING_V2).select('PROCESO_ESTANDARIZADO').unique().collect()
    .with_columns([
        pl.col('PROCESO_ESTANDARIZADO').cast(pl.Int64).alias('ANIO'),
        pl.lit(None, dtype=pl.Int64).alias('SEMESTRE'),
        pl.col('PROCESO_ESTANDARIZADO').cast(pl.String).alias('LABEL_PERIODO'),
        pl.lit('ANUAL').alias('TIPO_PERIODO'),
    ])
    .select(['ANIO', 'SEMESTRE', 'LABEL_PERIODO', 'TIPO_PERIODO'])
)
dim_periodo = (
    pl.concat([dim_periodo_sem, dim_periodo_an])
    .unique(subset=['ANIO', 'SEMESTRE'])
    .sort('ANIO', 'SEMESTRE')
    .with_row_index('SK_Periodo', offset=1)
)
dim_periodo.write_parquet(GOLD / 'dim_periodo.parquet')
print(f'DimPeriodo: {dim_periodo.height:,} filas → dim_periodo.parquet')
print(dim_periodo)

In [ ]:
# --- DimUbicacion: DEPARTAMENTO + PROVINCIA (FILIAL en ING, LOCAL en MAT) + Region_Sur ---
dim_ubicacion = (
    pl.concat([
        pl.scan_parquet(ING_V2).select([
            pl.col('DEPARTAMENTO_FILIAL').alias('DEPARTAMENTO'),
            pl.col('PROVINCIA_FILIAL').alias('PROVINCIA'),
        ]).unique().collect(),
        pl.scan_parquet(MAT_V2).select([
            pl.col('DEPARTAMENTO_LOCAL').alias('DEPARTAMENTO'),
            pl.col('PROVINCIA_LOCAL').alias('PROVINCIA'),
        ]).unique().collect(),
    ])
    .unique()
    .unique(subset=['DEPARTAMENTO', 'PROVINCIA'])
    .sort('DEPARTAMENTO', 'PROVINCIA')
    .with_columns(pl.col('DEPARTAMENTO').is_in(DEPARTAMENTOS_SUR).alias('Region_Sur'))
    .with_row_index('SK_Ubicacion', offset=1)
)
dim_ubicacion.write_parquet(GOLD / 'dim_ubicacion.parquet')
print(f'DimUbicacion: {dim_ubicacion.height:,} filas → dim_ubicacion.parquet')

# --- DimLocal: solo Matriculados ---
dim_local = (
    pl.scan_parquet(MAT_V2)
    .select(['CODIGO_LOCAL', 'DEPARTAMENTO_LOCAL', 'PROVINCIA_LOCAL', 'DISTRITO_LOCAL', 'ES_LOCAL_PRINCIPAL', 'CODIGO_UBIGEO_INEI_LOCAL'])
    .unique()
    .collect()
    .unique(subset=['CODIGO_LOCAL'])
    .sort('CODIGO_LOCAL')
    .with_row_index('SK_Local', offset=1)
)
dim_local.write_parquet(GOLD / 'dim_local.parquet')
print(f'DimLocal: {dim_local.height:,} filas → dim_local.parquet')
print()
print('Resumen de dimensiones generadas:')
for f in sorted(GOLD.glob('dim_*.parquet')):
    d = pl.scan_parquet(f)
    print(f'  {f.name:<26} {d.select(pl.len()).collect().item():>7,} filas')

---
## SECCIÓN 3 · Construcción de hechos Gold (con FKs)

Los hechos **sustituyen las claves naturales por FKs** (viajo a los `SK_*` de las dimensiones mediante joins), conservando los atributos degenerados (`SEXO`, `EDAD`, `NACIONALIDAD`, `Region_Sur`) y `GUID_PERSONA` como medida de conteo.

- **FactIngresantes** (3.12 M filas, en memoria): FK_Universidad, FK_Programa, FK_Periodo (anual), FK_Ubicacion.
- **FactMatriculados** (17.37 M filas, **streaming** con `sink_parquet`): + FK_Local.

Se guardan en `data/Gold/`.

In [ ]:
# --- FactIngresantes: claves → FK (dataset pequeño, en memoria) ---
fact_ing = (
    ing.lazy()
    .join(dim_univ.select(['CODIGO_INEI', 'SK_Universidad']).lazy(), on='CODIGO_INEI', how='left')
    .join(dim_prog.select(['CODIGO_SIU_PROGRAMA', 'SK_Programa']).lazy(), on='CODIGO_SIU_PROGRAMA', how='left')
    .join(
        dim_periodo.filter(pl.col('SEMESTRE').is_null()).select(['ANIO', 'SK_Periodo']).lazy(),
        left_on=pl.col('PROCESO_ESTANDARIZADO').cast(pl.Int64),
        right_on='ANIO',
        how='left',
    )
    .join(
        dim_ubicacion.select(['DEPARTAMENTO', 'PROVINCIA', 'SK_Ubicacion']).lazy(),
        left_on=['DEPARTAMENTO_FILIAL', 'PROVINCIA_FILIAL'],
        right_on=['DEPARTAMENTO', 'PROVINCIA'],
        how='left',
    )
    .select([
        'SK_Universidad', 'SK_Programa', 'SK_Periodo', 'SK_Ubicacion',
        'GUID_PERSONA', 'SEXO', 'EDAD', 'NACIONALIDAD', 'Region_Sur',
    ])
    .rename({
        'SK_Universidad': 'FK_Universidad',
        'SK_Programa': 'FK_Programa',
        'SK_Periodo': 'FK_Periodo',
        'SK_Ubicacion': 'FK_Ubicacion',
    })
    .collect()
)
fact_ing.write_parquet(GOLD / 'fact_ingresantes.parquet')
print(f'FactIngresantes: {fact_ing.height:,} filas · {len(fact_ing.columns)} cols → fact_ingresantes.parquet')
print(fact_ing.head(3))

In [ ]:
# --- FactMatriculados: claves → FK (streaming, sin OOM) ---
fact_mat = (
    pl.scan_parquet(MAT_V2)
    .with_columns([
        pl.col('PERIODO_ESTANDARIZADO').str.slice(0, 4).cast(pl.Int64).alias('ANIO'),
        pl.col('PERIODO_ESTANDARIZADO').str.slice(5, 1).cast(pl.Int64).alias('SEMESTRE'),
    ])
    .join(dim_univ.select(['CODIGO_INEI', 'SK_Universidad']).lazy(), on='CODIGO_INEI', how='left')
    .join(dim_prog.select(['CODIGO_SIU_PROGRAMA', 'SK_Programa']).lazy(), on='CODIGO_SIU_PROGRAMA', how='left')
    .join(dim_periodo.select(['ANIO', 'SEMESTRE', 'SK_Periodo']).lazy(), on=['ANIO', 'SEMESTRE'], how='left')
    .join(
        dim_ubicacion.select(['DEPARTAMENTO', 'PROVINCIA', 'SK_Ubicacion']).lazy(),
        left_on=['DEPARTAMENTO_LOCAL', 'PROVINCIA_LOCAL'],
        right_on=['DEPARTAMENTO', 'PROVINCIA'],
        how='left',
    )
    .join(dim_local.select(['CODIGO_LOCAL', 'SK_Local']).lazy(), on='CODIGO_LOCAL', how='left')
    .select([
        'SK_Universidad', 'SK_Programa', 'SK_Periodo', 'SK_Ubicacion', 'SK_Local',
        'GUID_PERSONA', 'SEXO', 'EDAD', 'NACIONALIDAD', 'Region_Sur',
    ])
    .rename({
        'SK_Universidad': 'FK_Universidad',
        'SK_Programa': 'FK_Programa',
        'SK_Periodo': 'FK_Periodo',
        'SK_Ubicacion': 'FK_Ubicacion',
        'SK_Local': 'FK_Local',
    })
)

GOLD_FACT_MAT = GOLD / 'fact_matriculados.parquet'
GOLD_FACT_MAT.unlink(missing_ok=True)
try:
    fact_mat.sink_parquet(GOLD_FACT_MAT)
    print('FactMatriculados: guardado con sink_parquet (streaming).')
except Exception as e:
    print(f'sink_parquet falló ({type(e).__name__}); usando collect(engine="streaming").write_parquet')
    fact_mat.collect(engine='streaming').write_parquet(GOLD_FACT_MAT)
print(f'→ {GOLD_FACT_MAT.name}')

---
## SECCIÓN 4 · Validación del modelo Gold

Se verifica que:
- **A** · Todas las tablas (`dim_*.parquet`, `fact_*.parquet`) existen en `data/Gold/`.
- **B** · Conteos de hechos == conteos de las fuentes V2.
- **C** · Integridad referencial: 0 FKs nulos y 0 FKs huérfanos (cada FK existe en su dimensión).
- **D** · Ejemplo de consulta sobre el modelo (matriculados por universidad y año).

In [ ]:
print('VALIDACIÓN DEL MODELO GOLD')
print('=' * 78)

# A) Tablas generadas en data/Gold/
print('A) Tablas generadas en data/Gold/:')
for f in sorted(GOLD.glob('*.parquet')):
    d = pl.scan_parquet(f)
    n = d.select(pl.len()).collect().item()
    cols = d.collect_schema().names()
    print(f'  {f.name:<28} {n:>10,} filas · {len(cols)} cols · {cols[0]}...')
print()

# B) Conteos de hechos vs fuentes
print('B) Conteos de hechos vs fuentes:')
ing_n = pl.scan_parquet(ING_V2).select(pl.len()).collect().item()
mat_n_src = pl.scan_parquet(MAT_V2).select(pl.len()).collect().item()
fact_ing_n = pl.scan_parquet(GOLD / 'fact_ingresantes.parquet').select(pl.len()).collect().item()
fact_mat_n = pl.scan_parquet(GOLD / 'fact_matriculados.parquet').select(pl.len()).collect().item()
print(f'  FactIngresantes {fact_ing_n:,} == Ingresantes V2 {ing_n:,} → {fact_ing_n == ing_n}')
print(f'  FactMatriculados {fact_mat_n:,} == Matriculados V2 {mat_n_src:,} → {fact_mat_n == mat_n_src}')
print()

# C) Integridad referencial (nulos y huérfanos por FK)
print('C) Integridad referencial (nulos y huérfanos por FK):')
fk_map = {
    'FK_Universidad': ('dim_universidad.parquet', 'SK_Universidad'),
    'FK_Programa': ('dim_programa.parquet', 'SK_Programa'),
    'FK_Periodo': ('dim_periodo.parquet', 'SK_Periodo'),
    'FK_Ubicacion': ('dim_ubicacion.parquet', 'SK_Ubicacion'),
    'FK_Local': ('dim_local.parquet', 'SK_Local'),
}
for hecho_nombre in ['fact_ingresantes.parquet', 'fact_matriculados.parquet']:
    hecho = pl.scan_parquet(GOLD / hecho_nombre)
    fks = [c for c in hecho.collect_schema().names() if c.startswith('FK_')]
    for fk in fks:
        dim_archivo, sk = fk_map[fk]
        dim = pl.scan_parquet(GOLD / dim_archivo)
        nulos = hecho.select(pl.col(fk).null_count()).collect().item()
        huerfanos = (
            hecho.select(fk).unique()
            .join(dim.select(sk).unique(), left_on=fk, right_on=sk, how='anti')
            .select(pl.len())
            .collect()
            .item()
        )
        ok = (nulos == 0) and (huerfanos == 0)
        print(f'  {hecho_nombre}.{fk}: nulos={nulos:,} · huérfanos={huerfanos:,} → {"OK" if ok else "FALLO"}')
print()

# D) Ejemplo de consulta sobre el modelo (matriculados por universidad y año)
print('D) Ejemplo: total de matriculados por universidad y año (top 8):')
query = (
    pl.scan_parquet(GOLD / 'fact_matriculados.parquet')
    .join(pl.scan_parquet(GOLD / 'dim_universidad.parquet'), left_on='FK_Universidad', right_on='SK_Universidad')
    .join(pl.scan_parquet(GOLD / 'dim_periodo.parquet'), left_on='FK_Periodo', right_on='SK_Periodo')
    .group_by(['NOMBRE_ENTIDAD', 'ANIO'])
    .len()
    .sort('len', descending=True)
    .head(8)
)
print(query.collect())
print('=' * 78)
print('Modelo Gold generado y validado.')

In [ ]:
# Liberar memoria de las estructuras grandes
del ing, mat, fact_ing, dim_univ, dim_prog, dim_periodo, dim_ubicacion, dim_local
gc.collect()
print(f'RSS final: {rss_actual_gb():.2f} GB')
print('OK: Capa Gold materializada y validada.')